# TRPO and PPO — interactive companion

Companion to [Post 2c: TRPO and PPO](../posts/02c-trpo-and-ppo.qmd).
A2C plus one fix — limit how much the policy can change per update.
The result is the workhorse algorithm of modern RL.

**What you'll do (≈ 20 minutes):**
1. Visualize the clipped surrogate $L^{\text{CLIP}}$ — see the asymmetric clip in action.
2. Train PPO with different numbers of epochs per batch.
3. Compare A2C vs PPO at aggressive learning rates — watch A2C collapse, PPO stay stable.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.mdp import TwoGoalGridWorld
from nano_agents.policy_gradient import (
    SoftmaxPolicy, TabularBaseline,
    train_a2c, train_ppo,
)

## 1. The clipped surrogate

For each (state, action, advantage) sample, define the importance ratio
$r(\theta) = \pi_\theta(a\mid s) / \pi_{\theta_{\rm old}}(a\mid s)$.
The PPO objective is

$$
L^{\mathrm{CLIP}}(\theta) = \mathbb{E}\!\left[\min\!\left(r\,A,\;\mathrm{clip}(r, 1{-}\epsilon, 1{+}\epsilon)\,A\right)\right].
$$

The min creates a zero-gradient region when the policy strays past the
clip in the unhelpful direction. Visualize it:

In [ ]:
eps = 0.2
r_grid = np.linspace(0.0, 2.0, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, A in zip(axes, [+1.0, -1.0]):
    unclipped = r_grid * A
    clipped = np.clip(r_grid, 1 - eps, 1 + eps) * A
    L = np.minimum(unclipped, clipped)
    ax.plot(r_grid, unclipped, "--", color="gray", label=r"$r\cdot A$ (unclipped)")
    ax.plot(r_grid, clipped, ":", color="C0", label=r"clip($r$)$\cdot A$")
    ax.plot(r_grid, L, color="C3", linewidth=2.5, label=r"$L^{\rm CLIP} = \min$")
    if A > 0:
        ax.axvspan(1 + eps, 2.0, color="#ffe5e5", alpha=0.5, label="zero gradient")
    else:
        ax.axvspan(0, 1 - eps, color="#ffe5e5", alpha=0.5, label="zero gradient")
    ax.axvline(1.0, color="black", linewidth=0.6, alpha=0.5)
    ax.set_xlabel("importance ratio $r$")
    ax.set_title(f"A = {A}"); ax.legend(loc="best", fontsize=9)
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The shaded region is where the gradient is zero — the policy can't move
further in the unhelpful direction. Crucially, the clip is **asymmetric**:
the policy can still move *back toward* $r = 1$ freely.

### Try this
- Set `eps = 0.5` (loose clip). The shaded region shrinks. With very
  loose clips, PPO is essentially vanilla policy gradient.
- Set `eps = 0.05` (tight clip). Almost everything past $r = 1.05$ has
  zero gradient — very conservative updates.

## 2. The multi-epochs trick

PPO's win over A2C is that you can do **many gradient steps per batch
of data**. With vanilla policy gradient, the second update would compute
the wrong gradient (the policy has changed). The clip makes those extra
updates safe.

In [ ]:
env = TwoGoalGridWorld(slip=0.0, step_reward=-0.04)
gamma = 0.95
n_seeds = 3

# Fix the total sample budget across configurations.
n_trajs = 16
target_transitions = 25_000
avg_traj_len = 12

configs = [
    ("PPO, 1 epoch",  1),
    ("PPO, 4 epochs", 4),
    ("PPO, 10 epochs", 10),
]
results = {}
for name, n_epochs in configs:
    n_iter = target_transitions // (n_trajs * avg_traj_len)
    returns_avg = np.zeros(n_iter)
    for seed in range(n_seeds):
        policy = SoftmaxPolicy(env.nS, env.nA)
        baseline = TabularBaseline(env.nS)
        h = train_ppo(env, policy, baseline,
                      n_iterations=n_iter,
                      n_trajectories_per_iter=n_trajs,
                      n_epochs=n_epochs,
                      lr_actor=0.05, lr_critic=0.2,
                      gamma=gamma, lam=0.95, eps=0.2,
                      rng=np.random.default_rng(seed))
        returns_avg += h["returns"] / n_seeds
    results[name] = returns_avg

transitions = np.arange(1, len(returns_avg) + 1) * n_trajs * avg_traj_len
for name, ret in results.items():
    plt.plot(transitions, ret, label=name)
plt.xlabel("transitions seen (approx.)")
plt.ylabel(f"mean return per iteration (avg over {n_seeds} seeds)")
plt.title("PPO sample efficiency vs epochs per batch")
plt.legend(); plt.grid(alpha=0.3); plt.show()

More epochs → faster convergence per environment step. The clip prevents
the extra updates from destabilizing the policy.

### Try this
- Add a `n_epochs = 32` run. Does it keep improving, or plateau?
- Set `eps = 0.05` and re-run. Tighter clip means each epoch moves the
  policy less, so more epochs are needed.

## 3. The big result: PPO is lr-robust

This is the headline finding of Post 2c. Vanilla policy gradient and A2C
both collapse catastrophically at large learning rates. PPO doesn't.

In [ ]:
lrs = [0.01, 0.05, 0.1, 0.3, 1.0, 2.0]
n_episodes_a2c = 1500
n_iter_ppo = n_episodes_a2c // n_trajs

final_a2c, final_ppo = {}, {}
for lr in lrs:
    a2c_finals, ppo_finals = [], []
    for seed in range(n_seeds):
        # A2C
        policy = SoftmaxPolicy(env.nS, env.nA)
        baseline = TabularBaseline(env.nS)
        h = train_a2c(env, policy, baseline, n_episodes=n_episodes_a2c,
                       lr_actor=lr, lr_critic=0.2,
                       gamma=gamma, lam=0.95,
                       rng=np.random.default_rng(seed))
        a2c_finals.append(float(np.mean(h["returns"][-100:])))
        # PPO
        policy = SoftmaxPolicy(env.nS, env.nA)
        baseline = TabularBaseline(env.nS)
        h = train_ppo(env, policy, baseline, n_iterations=n_iter_ppo,
                       n_trajectories_per_iter=n_trajs, n_epochs=4,
                       lr_actor=lr, lr_critic=0.2,
                       gamma=gamma, lam=0.95, eps=0.2,
                       rng=np.random.default_rng(seed))
        ppo_finals.append(float(np.mean(h["returns"][-20:])))
    final_a2c[lr] = (np.mean(a2c_finals), np.std(a2c_finals))
    final_ppo[lr] = (np.mean(ppo_finals), np.std(ppo_finals))

m_a, s_a = zip(*final_a2c.values())
m_p, s_p = zip(*final_ppo.values())
plt.errorbar(lrs, m_a, yerr=s_a, fmt="o-", label="A2C", color="C3", capsize=4)
plt.errorbar(lrs, m_p, yerr=s_p, fmt="s-", label="PPO", color="C0", capsize=4)
plt.xscale("log"); plt.xlabel("actor learning rate")
plt.ylabel(f"final return ({n_seeds} seeds, mean ± std)")
plt.title("PPO is lr-robust; A2C catastrophically fails at lr=2")
plt.legend(); plt.grid(which="both", alpha=0.3); plt.show()

print("\\nA2C final returns:")
for lr, (m, s) in final_a2c.items(): print(f"  lr={lr}: {m:.2f} ± {s:.2f}")
print("\\nPPO final returns:")
for lr, (m, s) in final_ppo.items(): print(f"  lr={lr}: {m:.2f} ± {s:.2f}")

**Headline result**: PPO is monotonically stable across two orders of magnitude
of learning rate. A2C catastrophically fails at lr = 2.0. This is the central
practical reason PPO became the default workhorse — you can use much wider
hyperparameter ranges without disaster.

In LLM RLHF this matters enormously: you can't easily re-tune per problem,
so robustness across hyperparameters is essential.

### Try this
- Set `eps = 0.05` for PPO (tight clip). At `lr=2.0`, PPO is now *very*
  constrained and learns slowly — what's its final return?
- Set `eps = 1.0` (essentially no clip). Does PPO now break at high lr
  like A2C?

## What's next

PPO + GAE + reward model = RLHF. The final piece of the policy-gradient
puzzle: how do *preferences* become *rewards*, and what's the elegant
algorithm that skips the reward model entirely?

Open [`02d-rlhf-dpo-grpo.ipynb`](02d-rlhf-dpo-grpo.ipynb).